# PredictStock — Inventory Optimization

This notebook compares current inventory with predicted demand and generates stock status and replenishment recommendations.

### Workflow
Current Stock → Predicted Demand → Required Stock → Stock Gap → Replenishment Recommendation → Stock Status

## <span style="color:red">SECTION 1 — SETUP & LOAD FORECAST</span>

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Project paths
PROJECT_ROOT = Path(r"C:\Users\lenovo\OneDrive\Music\PredictStock")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

# Final forecast generated in 04_forecasting.ipynb
FINAL_FORECAST_PATH = PROCESSED_DIR / "final_demand_forecast.parquet"

# Load final demand forecast
forecast_df = pd.read_parquet(FINAL_FORECAST_PATH)

print("FINAL DEMAND FORECAST")
print("-" * 40)
print("Shape:", forecast_df.shape)
print("Columns:", list(forecast_df.columns))
print("\nFirst 5 rows:")
display(forecast_df.head())

FINAL DEMAND FORECAST
----------------------------------------
Shape: (13447, 6)
Columns: ['item_id', 'store_id', 'pair_code', 'predicted_demand_7d', 'predicted_demand_30d', 'predicted_demand_90d']

First 5 rows:


,item_id,store_id,pair_code,predicted_demand_7d,predicted_demand_30d,predicted_demand_90d
0,001829cb707d,1,1,11.627809,53.971949,173.446545
1,001829cb707d,4,4,12.671307,60.146659,170.991845
2,001b302a9387,1,5,33.804674,156.471760,378.328813
3,001b302a9387,4,8,16.614141,93.921334,239.791532
4,001dadca6678,1,9,3.216114,13.083376,34.122427


## <span style="color:red">SECTION 2 — FORECAST DATA VALIDATION</span>

In [2]:
print("FORECAST DATA VALIDATION")
print("-" * 40)

# Basic uniqueness checks
print("Unique item-store pairs:", 
      forecast_df[["item_id", "store_id"]].drop_duplicates().shape[0])

print("Total forecast rows:", len(forecast_df))

print("\nUnique stores:", forecast_df["store_id"].nunique())
print("Stores:", sorted(forecast_df["store_id"].unique()))

print("\nMissing values:")
print(forecast_df.isna().sum())

print("\nDuplicate item-store pairs:",
      forecast_df.duplicated(["item_id", "store_id"]).sum())

FORECAST DATA VALIDATION
----------------------------------------
Unique item-store pairs: 13447
Total forecast rows: 13447

Unique stores: 4
Stores: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Missing values:
item_id                 0
store_id                0
pair_code               0
predicted_demand_7d     0
predicted_demand_30d    0
predicted_demand_90d    0
dtype: int64

Duplicate item-store pairs: 0


In [3]:
# FORECAST VALUE SANITY CHECK

forecast_columns = [
    "predicted_demand_7d",
    "predicted_demand_30d",
    "predicted_demand_90d"
]

print("FORECAST VALUE SANITY CHECK")
print("-" * 40)

for col in forecast_columns:
    print(f"\n{col}")
    print("  Minimum :", forecast_df[col].min())
    print("  Maximum :", forecast_df[col].max())
    print("  Mean    :", forecast_df[col].mean())
    print("  Median  :", forecast_df[col].median())
    print("  Zeros   :", (forecast_df[col] == 0).sum())
    print("  Negative:", (forecast_df[col] < 0).sum())

print("\nForecast ordering checks:")
print(
    "30D >= 7D:",
    (forecast_df["predicted_demand_30d"] >= forecast_df["predicted_demand_7d"]).mean() * 100,
    "%"
)

print(
    "90D >= 30D:",
    (forecast_df["predicted_demand_90d"] >= forecast_df["predicted_demand_30d"]).mean() * 100,
    "%"
)

FORECAST VALUE SANITY CHECK
----------------------------------------

predicted_demand_7d
  Minimum : 0.0
  Maximum : 10636.783551261526
  Mean    : 28.223875232447803
  Median  : 7.884959826523465
  Zeros   : 92
  Negative: 0

predicted_demand_30d
  Minimum : 0.0
  Maximum : 44311.76061828587
  Mean    : 124.75537370128473
  Median  : 37.08695576531805
  Zeros   : 64
  Negative: 0

predicted_demand_90d
  Minimum : 0.0
  Maximum : 68624.0153970278
  Mean    : 348.729597037848
  Median  : 111.67197720784294
  Zeros   : 72
  Negative: 0

Forecast ordering checks:
30D >= 7D: 99.53893061649438 %
90D >= 30D: 99.24146649810368 %


## <span style="color:red">SECTION 3 — CURRENT INVENTORY INPUT</span>

### Current Inventory Template

- Created an inventory template using the same **13,447 item-store pairs** as the demand forecast.
- Columns:
  - `item_id`
  - `store_id`
  - `pair_code`
  - `current_stock`
- `current_stock` is currently empty (`NaN`) because we do not have real inventory data yet.
- This template defines the structure that the future **current-stock CSV upload** will follow.
- For the actual user-upload CSV, `pair_code` will not be required; matching will use **`item_id + store_id`**.
- Next step: define the **required-stock and replenishment logic** using the 30-day demand forecast.

In [4]:
# Create a template from the exact item-store pairs
inventory_template = forecast_df[
    ["item_id", "store_id", "pair_code"]
].copy()

# Current stock is intentionally left empty for now.
# We will generate realistic demo values after defining
# the inventory optimization logic.
inventory_template["current_stock"] = np.nan

print("CURRENT INVENTORY TEMPLATE")
print("-" * 40)

print("Shape:", inventory_template.shape)
print("Columns:", list(inventory_template.columns))

print("\nUnique item-store pairs:",
      inventory_template[["item_id", "store_id"]].drop_duplicates().shape[0])

print("\nMissing current_stock values:",
      inventory_template["current_stock"].isna().sum())

display(inventory_template.head())

CURRENT INVENTORY TEMPLATE
----------------------------------------
Shape: (13447, 4)
Columns: ['item_id', 'store_id', 'pair_code', 'current_stock']

Unique item-store pairs: 13447

Missing current_stock values: 13447


,item_id,store_id,pair_code,current_stock
0,001829cb707d,1,1,NaN
1,001829cb707d,4,4,NaN
2,001b302a9387,1,5,NaN
3,001b302a9387,4,8,NaN
4,001dadca6678,1,9,NaN


## <span style="color:red">SECTION 4 — REQUIRED STOCK & REPLENISHMENT LOGIC</span>

# SECTION 4 — REQUIRED STOCK & REPLENISHMENT LOGIC

The 30-day demand forecast is used as the primary inventory planning horizon.

Required Stock = Predicted 30-Day Demand + Safety Stock

Safety Stock provides a buffer against demand variation and uncertainty.

Replenishment Quantity = max(Required Stock - Current Stock, 0)

The resulting stock gap is used to classify inventory as:
- Critical
- Low
- Sufficient

In [5]:
# REQUIRED STOCK & REPLENISHMENT LOGIC

# Use 20% of predicted 30-day demand as an initial safety buffer.
# This is a project-level assumption and can be refined later.
SAFETY_BUFFER_RATE = 0.20

forecast_df["safety_stock"] = (
    forecast_df["predicted_demand_30d"] * SAFETY_BUFFER_RATE
)

forecast_df["required_stock"] = (
    forecast_df["predicted_demand_30d"] +
    forecast_df["safety_stock"]
)

print("REQUIRED STOCK CALCULATION")
print("-" * 40)
print("Primary horizon : 30 days")
print("Safety buffer   :", SAFETY_BUFFER_RATE * 100, "%")

print("\nRequired stock summary:")
print(forecast_df["required_stock"].describe())

display(
    forecast_df[
        [
            "item_id",
            "store_id",
            "predicted_demand_30d",
            "safety_stock",
            "required_stock"
        ]
    ].head()
)

REQUIRED STOCK CALCULATION
----------------------------------------
Primary horizon : 30 days
Safety buffer   : 20.0 %

Required stock summary:
count    13447.000000
mean       149.706448
std        811.101829
min          0.000000
25%         15.095655
50%         44.504347
75%        121.304242
max      53174.112742
Name: required_stock, dtype: float64


,item_id,store_id,predicted_demand_30d,safety_stock,required_stock
0,001829cb707d,1,53.971949,10.794390,64.766338
1,001829cb707d,4,60.146659,12.029332,72.175990
2,001b302a9387,1,156.471760,31.294352,187.766112
3,001b302a9387,4,93.921334,18.784267,112.705601
4,001dadca6678,1,13.083376,2.616675,15.700051


## <span style="color:red">SECTION 5 — CREATE DEMO CURRENT INVENTORY</span>

In [6]:
np.random.seed(42)

# Generate a stock level between 30% and 130% of required stock
stock_factor = np.random.uniform(
    0.30,
    1.30,
    size=len(forecast_df)
)

forecast_df["current_stock"] = (
    forecast_df["required_stock"] * stock_factor
)

# Round inventory to whole units
forecast_df["current_stock"] = (
    forecast_df["current_stock"].round().astype(int)
)

print("DEMO CURRENT INVENTORY")
print("-" * 40)

print("Rows:", len(forecast_df))
print("Minimum current stock:", forecast_df["current_stock"].min())
print("Maximum current stock:", forecast_df["current_stock"].max())
print("Mean current stock:", forecast_df["current_stock"].mean())
print("Zero-stock products:", (forecast_df["current_stock"] == 0).sum())

display(
    forecast_df[
        [
            "item_id",
            "store_id",
            "current_stock",
            "predicted_demand_30d",
            "required_stock"
        ]
    ].head()
)

DEMO CURRENT INVENTORY
----------------------------------------
Rows: 13447
Minimum current stock: 0
Maximum current stock: 58758
Mean current stock: 121.35048709749387
Zero-stock products: 375


,item_id,store_id,current_stock,predicted_demand_30d,required_stock
0,001829cb707d,1,44,53.971949,64.766338
1,001829cb707d,4,90,60.146659,72.175990
2,001b302a9387,1,194,156.471760,187.766112
3,001b302a9387,4,101,93.921334,112.705601
4,001dadca6678,1,7,13.083376,15.700051


## <span style="color:red">SECTION 6 — STOCK GAP & REPLENISHMENT QUANTITY</span>

In [7]:
forecast_df["stock_gap"] = (
    forecast_df["required_stock"] -
    forecast_df["current_stock"]
)

forecast_df["replenishment_quantity"] = (
    forecast_df["stock_gap"].clip(lower=0)
)

print("STOCK GAP & REPLENISHMENT")
print("-" * 40)

print("Products needing replenishment:",
      (forecast_df["replenishment_quantity"] > 0).sum())

print("Products with sufficient stock:",
      (forecast_df["replenishment_quantity"] == 0).sum())

print("\nReplenishment quantity summary:")
print(forecast_df["replenishment_quantity"].describe())

display(
    forecast_df[
        [
            "item_id",
            "store_id",
            "current_stock",
            "required_stock",
            "stock_gap",
            "replenishment_quantity"
        ]
    ].head()
)

STOCK GAP & REPLENISHMENT
----------------------------------------
Products needing replenishment: 9496
Products with sufficient stock: 3951

Replenishment quantity summary:
count    13447.000000
mean        35.219744
std        150.263305
min          0.000000
25%          0.000000
50%          4.338546
75%         23.034101
max       8588.799488
Name: replenishment_quantity, dtype: float64


,item_id,store_id,current_stock,required_stock,stock_gap,replenishment_quantity
0,001829cb707d,1,44,64.766338,20.766338,20.766338
1,001829cb707d,4,90,72.175990,-17.824010,0.000000
2,001b302a9387,1,194,187.766112,-6.233888,0.000000
3,001b302a9387,4,101,112.705601,11.705601,11.705601
4,001dadca6678,1,7,15.700051,8.700051,8.700051


## <span style="color:red">SECTION 7 — STOCK STATUS CLASSIFICATION</span>

In [8]:
# Stock status is based on current stock relative to required stock.
# Critical : current stock < 50% of required stock
# Low      : 50% <= current stock < 100% of required stock
# Sufficient: current stock >= required stock

stock_ratio = np.where(
    forecast_df["required_stock"] > 0,
    forecast_df["current_stock"] / forecast_df["required_stock"],
    np.inf
)

forecast_df["stock_ratio"] = stock_ratio

forecast_df["stock_status"] = np.select(
    [
        forecast_df["stock_ratio"] < 0.50,
        forecast_df["stock_ratio"] < 1.00
    ],
    [
        "Critical",
        "Low"
    ],
    default="Sufficient"
)

print("STOCK STATUS CLASSIFICATION")
print("-" * 40)

print(forecast_df["stock_status"].value_counts())

print("\nStock status percentages:")
print(
    forecast_df["stock_status"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

display(
    forecast_df[
        [
            "item_id",
            "store_id",
            "current_stock",
            "required_stock",
            "stock_ratio",
            "stock_status",
            "replenishment_quantity"
        ]
    ].head(10)
)

STOCK STATUS CLASSIFICATION
----------------------------------------
stock_status
Low           6593
Sufficient    3951
Critical      2903
Name: count, dtype: int64

Stock status percentages:
stock_status
Low           49.03
Sufficient    29.38
Critical      21.59
Name: proportion, dtype: float64


,item_id,store_id,current_stock,required_stock,stock_ratio,stock_status,replenishment_quantity
0,001829cb707d,1,44,64.766338,0.679365,Low,20.766338
1,001829cb707d,4,90,72.175990,1.246952,Sufficient,0.000000
2,001b302a9387,1,194,187.766112,1.033200,Sufficient,0.000000
3,001b302a9387,4,101,112.705601,0.896140,Low,11.705601
4,001dadca6678,1,7,15.700051,0.445858,Critical,8.700051
5,001dadca6678,2,0,0.593562,0.000000,Critical,0.593562
6,002555e3c4fa,1,107,298.594302,0.358346,Critical,191.594302
7,002555e3c4fa,2,69,58.982478,1.169839,Sufficient,0.000000
8,002555e3c4fa,4,339,376.607874,0.900140,Low,37.607874
9,00274a69c705,2,66,65.513759,1.007422,Sufficient,0.000000


## <span style="color:red">SECTION 8 — FINAL REPLENISHMENT RECOMMENDATION</span>

In [9]:
# Inventory units must be whole numbers.
# Therefore, round replenishment quantities upward.

forecast_df["recommended_replenishment"] = (
    np.ceil(forecast_df["replenishment_quantity"])
    .astype(int)
)

# Create final inventory recommendation
inventory_recommendation = forecast_df[
    [
        "item_id",
        "store_id",
        "current_stock",
        "predicted_demand_30d",
        "safety_stock",
        "required_stock",
        "stock_gap",
        "recommended_replenishment",
        "stock_status"
    ]
].copy()

print("FINAL INVENTORY RECOMMENDATION")
print("-" * 40)

print("Total item-store pairs:", len(inventory_recommendation))

print(
    "Total units recommended for replenishment:",
    inventory_recommendation["recommended_replenishment"].sum()
)

print("\nRecommendation summary:")
print(
    inventory_recommendation[
        "recommended_replenishment"
    ].describe()
)

display(inventory_recommendation.head(10))

FINAL INVENTORY RECOMMENDATION
----------------------------------------
Total item-store pairs: 13447
Total units recommended for replenishment: 478489

Recommendation summary:
count    13447.000000
mean        35.583327
std        150.294034
min          0.000000
25%          0.000000
50%          5.000000
75%         24.000000
max       8589.000000
Name: recommended_replenishment, dtype: float64


,item_id,store_id,current_stock,predicted_demand_30d,safety_stock,required_stock,stock_gap,recommended_replenishment,stock_status
0,001829cb707d,1,44,53.971949,10.794390,64.766338,20.766338,21,Low
1,001829cb707d,4,90,60.146659,12.029332,72.175990,-17.824010,0,Sufficient
2,001b302a9387,1,194,156.471760,31.294352,187.766112,-6.233888,0,Sufficient
3,001b302a9387,4,101,93.921334,18.784267,112.705601,11.705601,12,Low
4,001dadca6678,1,7,13.083376,2.616675,15.700051,8.700051,9,Critical
5,001dadca6678,2,0,0.494635,0.098927,0.593562,0.593562,1,Critical
6,002555e3c4fa,1,107,248.828585,49.765717,298.594302,191.594302,192,Critical
7,002555e3c4fa,2,69,49.152065,9.830413,58.982478,-10.017522,0,Sufficient
8,002555e3c4fa,4,339,313.839895,62.767979,376.607874,37.607874,38,Low
9,00274a69c705,2,66,54.594799,10.918960,65.513759,-0.486241,0,Sufficient


## <span style="color:red"># SECTION 9 — INVENTORY OPTIMIZATION SUMMARY</span>

In [10]:
print("INVENTORY OPTIMIZATION SUMMARY")
print("-" * 40)

print("Total item-store pairs :", len(inventory_recommendation))

print("\nStock status:")
print(
    inventory_recommendation["stock_status"]
    .value_counts()
)

print("\nTotal current stock:",
      inventory_recommendation["current_stock"].sum())

print("Total required stock:",
      inventory_recommendation["required_stock"].sum())

print("Total recommended replenishment:",
      inventory_recommendation["recommended_replenishment"].sum())

print("\nReplenishment required for:",
      (inventory_recommendation["recommended_replenishment"] > 0).sum(),
      "item-store pairs"
)

print("No replenishment required for:",
      (inventory_recommendation["recommended_replenishment"] == 0).sum(),
      "item-store pairs"
)

display(
    inventory_recommendation[
        [
            "item_id",
            "store_id",
            "current_stock",
            "required_stock",
            "recommended_replenishment",
            "stock_status"
        ]
    ].sort_values(
        "recommended_replenishment",
        ascending=False
    ).head(10)
)

INVENTORY OPTIMIZATION SUMMARY
----------------------------------------
Total item-store pairs : 13447

Stock status:
stock_status
Low           6593
Sufficient    3951
Critical      2903
Name: count, dtype: int64

Total current stock: 1631800
Total required stock: 2013102.6121934108
Total recommended replenishment: 478489

Replenishment required for: 9496 item-store pairs
No replenishment required for: 3951 item-store pairs


,item_id,store_id,current_stock,required_stock,recommended_replenishment,stock_status
8191,9a7e315f3f42,4,19570,28158.799488,8589,Low
8188,9a7e315f3f42,1,15590,19008.927250,3419,Low
505,0973df3ff57f,4,2924,6319.784009,3396,Critical
9317,b0d24502fb66,2,6263,9349.246181,3087,Low
6134,7428830d55b6,1,1354,4256.133155,2903,Critical
502,0973df3ff57f,1,4511,7400.459167,2890,Low
11428,daf04b3ec4a3,1,1071,3480.941923,2410,Critical
2760,348e96430d27,1,1890,4225.073897,2336,Critical
6458,79fdc441b59f,1,1176,3448.833342,2273,Critical
12751,f1eb9c33424e,4,1728,3952.587284,2225,Critical


## <span style="color:red">SECTION 10 — SAVE THE DASHBOARD READY RESULT</span>

In [12]:
# ============================================================
# SECTION 10 — FINAL DASHBOARD-READY DATASET
# ============================================================

dashboard_df = forecast_df[
    [
        "item_id",
        "store_id",
        "predicted_demand_7d",
        "predicted_demand_30d",
        "predicted_demand_90d",
        "current_stock",
        "safety_stock",
        "required_stock",
        "stock_gap",
        "recommended_replenishment",
        "stock_ratio",
        "stock_status"
    ]
].copy()

# ------------------------------------------------------------
# Keep model/calculated values as decimals
# Physical inventory quantities remain whole numbers.
# ------------------------------------------------------------

# Current physical stock → whole units
dashboard_df["current_stock"] = (
    dashboard_df["current_stock"].astype(int)
)

# Recommended order quantity → whole units
dashboard_df["recommended_replenishment"] = (
    dashboard_df["recommended_replenishment"].astype(int)
)

# Keep model predictions and calculated requirements as decimals
dashboard_df["predicted_demand_7d"] = (
    dashboard_df["predicted_demand_7d"].round(2)
)

dashboard_df["predicted_demand_30d"] = (
    dashboard_df["predicted_demand_30d"].round(2)
)

dashboard_df["predicted_demand_90d"] = (
    dashboard_df["predicted_demand_90d"].round(2)
)

dashboard_df["safety_stock"] = (
    dashboard_df["safety_stock"].round(2)
)

dashboard_df["required_stock"] = (
    dashboard_df["required_stock"].round(2)
)

dashboard_df["stock_gap"] = (
    dashboard_df["stock_gap"].round(2)
)

# Export
FINAL_CSV_PATH = PROCESSED_DIR / "final_inventory_recommendation.csv"
FINAL_PARQUET_PATH = PROCESSED_DIR / "final_inventory_recommendation.parquet"

dashboard_df.to_csv(FINAL_CSV_PATH, index=False)
dashboard_df.to_parquet(FINAL_PARQUET_PATH, index=False)

print("FINAL DASHBOARD DATASET")
print("-" * 40)

print("Shape:", dashboard_df.shape)
print("Columns:", list(dashboard_df.columns))

print("\nCSV saved to:")
print(FINAL_CSV_PATH)

print("\nParquet saved to:")
print(FINAL_PARQUET_PATH)

display(dashboard_df.head())

FINAL DASHBOARD DATASET
----------------------------------------
Shape: (13447, 12)
Columns: ['item_id', 'store_id', 'predicted_demand_7d', 'predicted_demand_30d', 'predicted_demand_90d', 'current_stock', 'safety_stock', 'required_stock', 'stock_gap', 'recommended_replenishment', 'stock_ratio', 'stock_status']

CSV saved to:
C:\Users\lenovo\OneDrive\Music\PredictStock\data\processed\final_inventory_recommendation.csv

Parquet saved to:
C:\Users\lenovo\OneDrive\Music\PredictStock\data\processed\final_inventory_recommendation.parquet


,item_id,store_id,predicted_demand_7d,predicted_demand_30d,predicted_demand_90d,current_stock,safety_stock,required_stock,stock_gap,recommended_replenishment,stock_ratio,stock_status
0,001829cb707d,1,11.63,53.97,173.45,44,10.79,64.77,20.77,21,0.679365,Low
1,001829cb707d,4,12.67,60.15,170.99,90,12.03,72.18,-17.82,0,1.246952,Sufficient
2,001b302a9387,1,33.80,156.47,378.33,194,31.29,187.77,-6.23,0,1.033200,Sufficient
3,001b302a9387,4,16.61,93.92,239.79,101,18.78,112.71,11.71,12,0.896140,Low
4,001dadca6678,1,3.22,13.08,34.12,7,2.62,15.70,8.70,9,0.445858,Critical


### Required Stock & Replenishment Logic

- **Required Stock** = Predicted 30-Day Demand + Safety Stock
- **Safety Stock** = 20% of Predicted 30-Day Demand
- Therefore, **Required Stock = 1.20 × Predicted 30-Day Demand**
- Safety stock provides an additional buffer against demand uncertainty.
- **Replenishment Quantity** = max(Required Stock − Current Stock, 0)
- The system recommends replenishment only when current stock is below the required stock level.

## <span style="color:red">SECTION 11 — FINAL DATASET VALIDATION</span>

In [13]:
print("FINAL DATASET VALIDATION")
print("-" * 40)

csv_check = pd.read_csv(FINAL_CSV_PATH)
parquet_check = pd.read_parquet(FINAL_PARQUET_PATH)

print("CSV shape     :", csv_check.shape)
print("Parquet shape :", parquet_check.shape)

print("\nCSV missing values:")
print(csv_check.isna().sum())

print("\nDuplicate item-store pairs:",
      csv_check.duplicated(["item_id", "store_id"]).sum())

print("\nStock status distribution:")
print(csv_check["stock_status"].value_counts())

print("\nRecommended replenishment total:",
      csv_check["recommended_replenishment"].sum())

print("\nFinal inventory quantity data types:")
print(
    csv_check[
        [
            "predicted_demand_7d",
            "predicted_demand_30d",
            "predicted_demand_90d",
            "current_stock",
            "safety_stock",
            "required_stock",
            "stock_gap",
            "recommended_replenishment"
        ]
    ].dtypes
)

print("\nValidation complete.")

FINAL DATASET VALIDATION
----------------------------------------
CSV shape     : (13447, 12)
Parquet shape : (13447, 12)

CSV missing values:
item_id                      0
store_id                     0
predicted_demand_7d          0
predicted_demand_30d         0
predicted_demand_90d         0
current_stock                0
safety_stock                 0
required_stock               0
stock_gap                    0
recommended_replenishment    0
stock_ratio                  0
stock_status                 0
dtype: int64

Duplicate item-store pairs: 0

Stock status distribution:
stock_status
Low           6593
Sufficient    3951
Critical      2903
Name: count, dtype: int64

Recommended replenishment total: 478489

Final inventory quantity data types:
predicted_demand_7d          float64
predicted_demand_30d         float64
predicted_demand_90d         float64
current_stock                  int64
safety_stock                 float64
required_stock               float64
stock_gap       

## <span style="color:red">SECTION 12 — INVENTORY ANALYSIS</span>

In [14]:
# INVENTORY ANALYSIS

print("OVERALL INVENTORY STATUS ANALYSIS")
print("-" * 40)

status_counts = (
    dashboard_df["stock_status"]
    .value_counts()
    .reindex(["Critical", "Low", "Sufficient"])
)

status_percent = (
    status_counts / len(dashboard_df) * 100
).round(2)

print("Total item-store pairs:", len(dashboard_df))

print("\nStock Status:")
for status in status_counts.index:
    print(
        f"{status:12s}: {status_counts[status]:,} "
        f"({status_percent[status]}%)"
    )

print("\nTotal current stock:",
      dashboard_df["current_stock"].sum())

print("Total required stock:",
      dashboard_df["required_stock"].sum())

print("Total recommended replenishment:",
      dashboard_df["recommended_replenishment"].sum())

OVERALL INVENTORY STATUS ANALYSIS
----------------------------------------
Total item-store pairs: 13447

Stock Status:
Critical    : 2,903 (21.59%)
Low         : 6,593 (49.03%)
Sufficient  : 3,951 (29.38%)

Total current stock: 1631800
Total required stock: 2013102.7999999998
Total recommended replenishment: 478489


In [15]:
# STORE-WISE INVENTORY ANALYSIS

store_analysis = (
    dashboard_df
    .groupby("store_id")
    .agg(
        item_store_pairs=("item_id", "count"),
        current_stock=("current_stock", "sum"),
        required_stock=("required_stock", "sum"),
        recommended_replenishment=("recommended_replenishment", "sum")
    )
    .reset_index()
)

store_analysis["stock_coverage_percent"] = (
    store_analysis["current_stock"] /
    store_analysis["required_stock"].replace(0, np.nan) * 100
).round(2)

print("STORE-WISE INVENTORY ANALYSIS")
print("-" * 40)

display(store_analysis)

STORE-WISE INVENTORY ANALYSIS
----------------------------------------


,store_id,item_store_pairs,current_stock,required_stock,recommended_replenishment,stock_coverage_percent
0,1,5010,664475,812340.55,191167,81.80
1,2,1736,112136,145063.05,39222,77.30
2,3,914,154708,186503.13,40800,82.95
3,4,5787,700481,869196.07,207300,80.59


In [16]:
# TOP REPLENISHMENT REQUIREMENTS

print("TOP REPLENISHMENT REQUIREMENTS")
print("-" * 40)

top_replenishment = (
    dashboard_df[
        [
            "item_id",
            "store_id",
            "current_stock",
            "predicted_demand_30d",
            "required_stock",
            "recommended_replenishment",
            "stock_status"
        ]
    ]
    .sort_values(
        "recommended_replenishment",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)

display(top_replenishment)

TOP REPLENISHMENT REQUIREMENTS
----------------------------------------


,item_id,store_id,current_stock,predicted_demand_30d,required_stock,recommended_replenishment,stock_status
0,9a7e315f3f42,4,19570,23465.67,28158.80,8589,Low
1,9a7e315f3f42,1,15590,15840.77,19008.93,3419,Low
2,0973df3ff57f,4,2924,5266.49,6319.78,3396,Critical
3,b0d24502fb66,2,6263,7791.04,9349.25,3087,Low
4,7428830d55b6,1,1354,3546.78,4256.13,2903,Critical
5,0973df3ff57f,1,4511,6167.05,7400.46,2890,Low
6,daf04b3ec4a3,1,1071,2900.78,3480.94,2410,Critical
7,348e96430d27,1,1890,3520.89,4225.07,2336,Critical
8,79fdc441b59f,1,1176,2874.03,3448.83,2273,Critical
9,f1eb9c33424e,4,1728,3293.82,3952.59,2225,Critical


In [17]:
# CRITICAL STOCK ANALYSIS

print("CRITICAL STOCK ANALYSIS")
print("-" * 40)

critical_df = dashboard_df[
    dashboard_df["stock_status"] == "Critical"
].copy()

print("Critical item-store pairs:", len(critical_df))

print(
    "Total replenishment required for critical stock:",
    critical_df["recommended_replenishment"].sum()
)

print(
    "Average current stock:",
    round(critical_df["current_stock"].mean(), 2)
)

print(
    "Average required stock:",
    round(critical_df["required_stock"].mean(), 2)
)

print("\nTop 10 critical items by replenishment requirement:")

display(
    critical_df[
        [
            "item_id",
            "store_id",
            "current_stock",
            "required_stock",
            "stock_ratio",
            "recommended_replenishment"
        ]
    ]
    .sort_values(
        "recommended_replenishment",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)

CRITICAL STOCK ANALYSIS
----------------------------------------
Critical item-store pairs: 2903
Total replenishment required for critical stock: 232053
Average current stock: 52.3
Average required stock: 131.7

Top 10 critical items by replenishment requirement:


,item_id,store_id,current_stock,required_stock,stock_ratio,recommended_replenishment
0,0973df3ff57f,4,2924,6319.78,0.462674,3396
1,7428830d55b6,1,1354,4256.13,0.318129,2903
2,daf04b3ec4a3,1,1071,3480.94,0.307675,2410
3,348e96430d27,1,1890,4225.07,0.447329,2336
4,79fdc441b59f,1,1176,3448.83,0.340985,2273
5,f1eb9c33424e,4,1728,3952.59,0.437182,2225
6,0c6f8d9fd621,1,1457,3578.17,0.407192,2122
7,f1eb9c33424e,1,1837,3862.91,0.475548,2026
8,26dade6734b3,4,953,2912.43,0.327218,1960
9,f1eb9c33424e,3,918,2796.24,0.328298,1879


In [18]:
# REPLENISHMENT BY STOCK STATUS

print("REPLENISHMENT BY STOCK STATUS")
print("-" * 40)

status_replenishment = (
    dashboard_df
    .groupby("stock_status")
    .agg(
        item_store_pairs=("item_id", "count"),
        total_current_stock=("current_stock", "sum"),
        total_required_stock=("required_stock", "sum"),
        total_replenishment=("recommended_replenishment", "sum")
    )
    .reindex(["Critical", "Low", "Sufficient"])
    .reset_index()
)

display(status_replenishment)

REPLENISHMENT BY STOCK STATUS
----------------------------------------


,stock_status,item_store_pairs,total_current_stock,total_required_stock,total_replenishment
0,Critical,2903,151823,382322.71,232053
1,Low,6593,722561,965661.41,246436
2,Sufficient,3951,757416,665118.68,0


In [19]:
#  INVENTORY COVERAGE ANALYSIS 

print("INVENTORY COVERAGE ANALYSIS")
print("-" * 40)

# Exclude infinite ratios caused by zero required stock
finite_coverage = dashboard_df[
    np.isfinite(dashboard_df["stock_ratio"])
]["stock_ratio"]

print("Item-store pairs with finite coverage:", len(finite_coverage))
print("Item-store pairs with zero required stock:",
      (~np.isfinite(dashboard_df["stock_ratio"])).sum())

print("\nStock ratio statistics:")
display(finite_coverage.describe().to_frame("stock_ratio"))

print("\nCoverage interpretation:")
print("Critical  : Stock ratio < 0.50")
print("Low       : 0.50 <= Stock ratio < 1.00")
print("Sufficient: Stock ratio >= 1.00")

print("\nAverage finite stock ratio:",
      round(finite_coverage.mean(), 3))

print("Median finite stock ratio:",
      round(finite_coverage.median(), 3))

INVENTORY COVERAGE ANALYSIS
----------------------------------------
Item-store pairs with finite coverage: 13383
Item-store pairs with zero required stock: 64

Stock ratio statistics:


,stock_ratio
count,13383.000000
mean,0.782252
std,0.314115
min,0.000000
25%,0.532904
50%,0.785134
75%,1.041618
max,2.368155



Coverage interpretation:
Critical  : Stock ratio < 0.50
Low       : 0.50 <= Stock ratio < 1.00
Sufficient: Stock ratio >= 1.00

Average finite stock ratio: 0.782
Median finite stock ratio: 0.785


In [20]:
# REPLENISHMENT CONCENTRATION ANALYSIS

print("REPLENISHMENT CONCENTRATION ANALYSIS")
print("-" * 40)

total_replenishment = dashboard_df["recommended_replenishment"].sum()

critical_replenishment = dashboard_df.loc[
    dashboard_df["stock_status"] == "Critical",
    "recommended_replenishment"
].sum()

low_replenishment = dashboard_df.loc[
    dashboard_df["stock_status"] == "Low",
    "recommended_replenishment"
].sum()

print("Total recommended replenishment:",
      total_replenishment)

print(
    "Critical stock replenishment:",
    critical_replenishment,
    f"({critical_replenishment / total_replenishment * 100:.2f}%)"
)

print(
    "Low stock replenishment:",
    low_replenishment,
    f"({low_replenishment / total_replenishment * 100:.2f}%)"
)

print(
    "Sufficient stock replenishment:",
    dashboard_df.loc[
        dashboard_df["stock_status"] == "Sufficient",
        "recommended_replenishment"
    ].sum()
)

REPLENISHMENT CONCENTRATION ANALYSIS
----------------------------------------
Total recommended replenishment: 478489
Critical stock replenishment: 232053 (48.50%)
Low stock replenishment: 246436 (51.50%)
Sufficient stock replenishment: 0


In [21]:
# STORE-WISE REPLENISHMENT SHARE

print("STORE-WISE REPLENISHMENT SHARE")
print("-" * 40)

store_replenishment = (
    dashboard_df
    .groupby("store_id")
    .agg(
        item_store_pairs=("item_id", "count"),
        current_stock=("current_stock", "sum"),
        required_stock=("required_stock", "sum"),
        recommended_replenishment=("recommended_replenishment", "sum")
    )
    .reset_index()
)

total_replenishment = dashboard_df["recommended_replenishment"].sum()

store_replenishment["replenishment_share_percent"] = (
    store_replenishment["recommended_replenishment"]
    / total_replenishment * 100
).round(2)

display(
    store_replenishment.sort_values(
        "recommended_replenishment",
        ascending=False
    )
)

STORE-WISE REPLENISHMENT SHARE
----------------------------------------


,store_id,item_store_pairs,current_stock,required_stock,recommended_replenishment,replenishment_share_percent
3,4,5787,700481,869196.07,207300,43.32
0,1,5010,664475,812340.55,191167,39.95
2,3,914,154708,186503.13,40800,8.53
1,2,1736,112136,145063.05,39222,8.20


In [22]:
# OVERALL INVENTORY GAP

print("OVERALL INVENTORY GAP")
print("-" * 40)

total_current = dashboard_df["current_stock"].sum()
total_required = dashboard_df["required_stock"].sum()
total_gap = total_required - total_current

coverage_percent = (
    total_current / total_required * 100
)

print("Total current stock       :", total_current)
print("Total required stock      :", round(total_required, 2))
print("Overall inventory gap     :", round(total_gap, 2))
print("Overall stock coverage    :", round(coverage_percent, 2), "%")
print(
    "Recommended replenishment:",
    dashboard_df["recommended_replenishment"].sum()
)

OVERALL INVENTORY GAP
----------------------------------------
Total current stock       : 1631800
Total required stock      : 2013102.8
Overall inventory gap     : 381302.8
Overall stock coverage    : 81.06 %
Recommended replenishment: 478489


## <span style="color:red">Section 13 — Final Output & Conclusion</span>

In [23]:
# SECTION 13 — FINAL OUTPUT & CONCLUSION

print("FINAL PREDICTSTOCK INVENTORY RECOMMENDATION")
print("-" * 50)

print("Forecast horizon used for inventory planning : 30 days")
print("Safety buffer used                         : 20%")
print("Total item-store pairs analyzed            :", len(dashboard_df))
print("Total current stock                        :", dashboard_df["current_stock"].sum())
print("Total required stock                       :", round(dashboard_df["required_stock"].sum(), 2))
print("Total recommended replenishment            :", dashboard_df["recommended_replenishment"].sum())

print("\nSTOCK STATUS")
print("-" * 30)

for status in ["Critical", "Low", "Sufficient"]:
    count = (dashboard_df["stock_status"] == status).sum()
    percentage = count / len(dashboard_df) * 100
    print(f"{status:12s}: {count:,} ({percentage:.2f}%)")

print("\nRESULT")
print("-" * 30)
print(
    "The inventory optimization stage converts predicted demand "
    "into actionable replenishment recommendations."
)

FINAL PREDICTSTOCK INVENTORY RECOMMENDATION
--------------------------------------------------
Forecast horizon used for inventory planning : 30 days
Safety buffer used                         : 20%
Total item-store pairs analyzed            : 13447
Total current stock                        : 1631800
Total required stock                       : 2013102.8
Total recommended replenishment            : 478489

STOCK STATUS
------------------------------
Critical    : 2,903 (21.59%)
Low         : 6,593 (49.03%)
Sufficient  : 3,951 (29.38%)

RESULT
------------------------------
The inventory optimization stage converts predicted demand into actionable replenishment recommendations.


In [24]:
#  FINAL PROJECT CONCLUSION

print("PROJECT CONCLUSION")
print("-" * 50)

print(
    "PredictStock combines demand forecasting with inventory "
    "optimization to support product-level and store-level "
    "replenishment decisions."
)

print(
    "\nThe 30-day demand forecast is used as the primary "
    "inventory planning horizon."
)

print(
    "\nA 20% safety buffer is added to forecasted demand "
    "to calculate required stock."
)

print(
    "\nCurrent inventory is compared with required stock to "
    "calculate replenishment quantities and classify stock "
    "as Critical, Low, or Sufficient."
)

print(
    "\nThe final dashboard-ready dataset contains",
    len(dashboard_df),
    "item-store recommendations."
)

print(
    "\nTotal recommended replenishment:",
    dashboard_df["recommended_replenishment"].sum(),
    "units."
)

print(
    "\nThe resulting dataset can be used by the future "
    "PredictStock web dashboard for inventory monitoring, "
    "filtering, and replenishment decision support."
)

PROJECT CONCLUSION
--------------------------------------------------
PredictStock combines demand forecasting with inventory optimization to support product-level and store-level replenishment decisions.

The 30-day demand forecast is used as the primary inventory planning horizon.

A 20% safety buffer is added to forecasted demand to calculate required stock.

Current inventory is compared with required stock to calculate replenishment quantities and classify stock as Critical, Low, or Sufficient.

The final dashboard-ready dataset contains 13447 item-store recommendations.

Total recommended replenishment: 478489 units.

The resulting dataset can be used by the future PredictStock web dashboard for inventory monitoring, filtering, and replenishment decision support.
